# **General NLP pipeline**
* Data gathering
* Text cleaning
* Vectorization
* Model training
* Deployment

In [92]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [93]:
data=pd.read_csv('/content/train.txt',sep=';',header=None,names=['text','emotion'])
data.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


**Text cleaning**

**Check for missing values**

In [94]:
data.isnull().sum()

,0
text,0
emotion,0


**Label enocding of target**

In [95]:
data['emotion'].value_counts()

,count
emotion,
joy,5362
sadness,4666
anger,2159
fear,1937
love,1304
surprise,572


In [96]:
from sklearn.preprocessing import LabelEncoder

encoder=LabelEncoder()
data['emotion_encoded']=encoder.fit_transform(data[['emotion']])

In [97]:
data.head()

,text,emotion,emotion_encoded
0,i didnt feel humiliated,sadness,4
1,i can go from feeling so hopeless to so damned...,sadness,4
2,im grabbing a minute to post i feel greedy wrong,anger,0
3,i am ever feeling nostalgic about the fireplac...,love,3
4,i am feeling grouchy,anger,0


In [98]:
data[['emotion','emotion_encoded']].value_counts()

,,count
emotion,emotion_encoded,
joy,2,5362
sadness,4,4666
anger,0,2159
fear,1,1937
love,3,1304
surprise,5,572


**LowerCase conversion**

In [99]:
data['text']=data['text'].apply(lambda x: x.lower())
data.head()

,text,emotion,emotion_encoded
0,i didnt feel humiliated,sadness,4
1,i can go from feeling so hopeless to so damned...,sadness,4
2,im grabbing a minute to post i feel greedy wrong,anger,0
3,i am ever feeling nostalgic about the fireplac...,love,3
4,i am feeling grouchy,anger,0


**Remvoe punctuations: Usually done with ML and not with DL**

In [100]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [101]:
data['text']=data['text'].apply(remove_punc)

**Remove numbers**

In [102]:
def remove_num(txt):
  result=""
  for i in txt:
    if not i.isdigit():
      result=result + i
  return result

data['text']=data['text'].apply(remove_num)

**Remove emojis**

In [103]:
def remove_emojis(txt):
  result=""
  for i in txt:
    if i.isascii():
      result= result + i
  return result

data['text']=data['text'].apply(remove_emojis)

**Tokenize and remove stop words**

In [104]:
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [105]:
stopwords=set(stopwords.words('english'))

In [106]:
stopwords

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [107]:
def remove_stop_words(txt):
  words=txt.split()
  result=[]
  for i in words:
    if i not in stopwords:
      result.append(i)
  return ' '.join(result)

In [108]:
data['text']=data['text'].apply(remove_stop_words)

In [109]:
data.head()

,text,emotion,emotion_encoded
0,didnt feel humiliated,sadness,4
1,go feeling hopeless damned hopeful around some...,sadness,4
2,im grabbing minute post feel greedy wrong,anger,0
3,ever feeling nostalgic fireplace know still pr...,love,3
4,feeling grouchy,anger,0


**Train Test Split**

In [110]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(data['text'],data['emotion_encoded'],test_size=0.2,random_state=42)

In [111]:
print(X_train.shape)
print(X_test.shape)

(12800,)
(3200,)


**Vectorizaiton**

In [112]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer=CountVectorizer()
X_train_count=vectorizer.fit_transform(X_train)
X_test_count=vectorizer.transform(X_test)

In [113]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_tfidf=TfidfVectorizer()
X_train_tfidf=vectorizer_tfidf.fit_transform(X_train)
X_test_tfidf=vectorizer_tfidf.transform(X_test)


**Model**

In [114]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

models={
    'Logistic Regression':LogisticRegression(),
    'Naive Bayes':MultinomialNB()
}

In [115]:
from sklearn.metrics import classification_report
for i,j in models.items():
  j.fit(X_train_count,y_train)
  y_pred=j.predict(X_test_count)
  print(f"{i}\n", classification_report(y_test,y_pred))

Logistic Regression
               precision    recall  f1-score   support

           0       0.89      0.86      0.87       427
           1       0.86      0.84      0.85       397
           2       0.88      0.94      0.91      1021
           3       0.85      0.76      0.80       296
           4       0.93      0.93      0.93       946
           5       0.86      0.73      0.79       113

    accuracy                           0.89      3200
   macro avg       0.88      0.84      0.86      3200
weighted avg       0.89      0.89      0.89      3200

Naive Bayes
               precision    recall  f1-score   support

           0       0.89      0.63      0.74       427
           1       0.85      0.57      0.68       397
           2       0.73      0.96      0.83      1021
           3       0.93      0.27      0.41       296
           4       0.75      0.95      0.84       946
           5       1.00      0.05      0.10       113

    accuracy                           0.77

In [116]:
lr=LogisticRegression()
model=lr.fit(X_train_count,y_train)

In [117]:
import pickle

with open('model.pkl','wb') as f:
  pickle.dump(model,f)

with open('vectorizer.pkl','wb') as f:
  pickle.dump(vectorizer,f)

In [118]:
with open('model.pkl','rb') as f:
  predictor=pickle.load(f)

with open('vectorizer.pkl','rb') as fv:
  vectorizer=pickle.load(fv)

**Test**

In [123]:
data.head()

,text,emotion,emotion_encoded
0,didnt feel humiliated,sadness,4
1,go feeling hopeless damned hopeful around some...,sadness,4
2,im grabbing minute post feel greedy wrong,anger,0
3,ever feeling nostalgic fireplace know still pr...,love,3
4,feeling grouchy,anger,0


In [131]:
emotions_mapped={
    4:'sadness', 0:'anger',3:'love',5:'surprise',1:'fear',2:'joy'
}

In [137]:
test_data=["Hard work never disappoints","Hello, Sam is the Altman","The girl is looked dangerous","He just had an accident"]

test_data_vectorized=vectorizer.transform(test_data)
output=predictor.predict(test_data_vectorized)

for text,pred in zip(test_data,output):
  print(f"{text}->", emotions_mapped[pred])

Hard work never disappoints-> joy
Hello, Sam is the Altman-> joy
The girl is looked dangerous-> anger
He just had an accident-> sadness
